<a href="https://colab.research.google.com/github/S48avio/Tokenizers/blob/main/Character_Level_BytePair_Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### COMPLETE BPE TOKENIZER CLASS

In [7]:
from collections import defaultdict


class BPETokenizer:
    def __init__(self):
        self.merge_rules = []
        self.token_to_id = {}
        self.id_to_token = {}
        self.token_vocab = set()

    def _get_vocab(self, corpus: str) -> dict:
        vocab = defaultdict(int)
        for word in corpus.split():
            vocab[tuple(list(word) + ['</w>'])] += 1
        return dict(vocab)

    def _get_pair_counts(self, vocab: dict) -> dict:
        counts = defaultdict(int)
        for word_tuple, freq in vocab.items():
            for i in range(len(word_tuple) - 1):
                counts[(word_tuple[i], word_tuple[i+1])] += freq
        return dict(counts)

    def _merge_vocab(self, pair: tuple, vocab: dict) -> dict:
        a, b = pair
        merged = a + b
        new_vocab = {}
        for word_tuple, freq in vocab.items():
            new_word = []
            i = 0
            while i < len(word_tuple):
                if (i < len(word_tuple)-1
                        and word_tuple[i] == a
                        and word_tuple[i+1] == b):
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word_tuple[i])
                    i += 1
            new_vocab[tuple(new_word)] = freq
        return new_vocab

    def train(self, corpus: str, num_merges: int):
        vocab = self._get_vocab(corpus)

        # ✅ FIX 1: seed token vocab with ALL chars seen in corpus
        for word_tuple in vocab:
            for ch in word_tuple:
                self.token_vocab.add(ch)

        for _ in range(num_merges):
            pair_counts = self._get_pair_counts(vocab)
            if not pair_counts:
                break
            best = max(pair_counts, key=pair_counts.get)
            vocab = self._merge_vocab(best, vocab)
            self.merge_rules.append(best)
            self.token_vocab.add(best[0] + best[1])

        sorted_tokens = sorted(self.token_vocab)
        self.token_to_id = {t: i for i, t in enumerate(sorted_tokens)}
        self.id_to_token = {i: t for t, i in self.token_to_id.items()}

    def _encode_word(self, word: str) -> list[str]:
        # ✅ FIX 2: handle unknown chars — fall back to <unk> per char
        tokens = []
        for ch in list(word) + ['</w>']:
            if ch in self.token_to_id:
                tokens.append(ch)
            else:
                tokens.append('<unk>')   # unknown char

        for (a, b) in self.merge_rules:
            new_tokens = []
            i = 0
            while i < len(tokens):
                if (i < len(tokens)-1
                        and tokens[i] == a
                        and tokens[i+1] == b):
                    new_tokens.append(a + b)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        return tokens

    def encode(self, text: str) -> list[int]:
        ids = []
        for word in text.split():
            for tok in self._encode_word(word):
                if tok in self.token_to_id:
                    ids.append(self.token_to_id[tok])
                # <unk> tokens without an ID are still skipped here
                # — to handle properly, add '<unk>' to vocab during train
        return ids

    def decode(self, ids: list[int]) -> str:
        tokens = [self.id_to_token[i] for i in ids]

        # ✅ FIX 3: handle </w> fused into tokens like 'low</w>'
        words = []
        current_word = []
        for tok in tokens:
            if tok == '</w>':
                words.append(''.join(current_word))
                current_word = []
            elif tok.endswith('</w>'):
                current_word.append(tok[:-4])   # strip the fused </w>
                words.append(''.join(current_word))
                current_word = []
            else:
                current_word.append(tok)
        if current_word:            # last word had no </w> (shouldn't happen)
            words.append(''.join(current_word))

        return ' '.join(words)

    def tokenize(self, text: str) -> list[str]:
        result = []
        for word in text.split():
            result.extend(self._encode_word(word))
        return result


# ── Demo ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    corpus = "low lower newest widest lowest low low"

    tok = BPETokenizer()
    tok.train(corpus, num_merges=15)

    tests = [
        "low lowest lower",
        "newest widest",
        "the cat sat on the mat",    # all unknown chars — shows <unk> behavior
    ]

    for text in tests:
        tokens  = tok.tokenize(text)
        ids     = tok.encode(text)
        decoded = tok.decode(ids)
        print(f"Input   : '{text}'")
        print(f"Tokens  : {tokens}")
        print(f"Decoded : '{decoded}'")
        print()

Input   : 'low lowest lower'
Tokens  : ['low</w>', 'low', 'est</w>', 'lower</w>']
Decoded : 'low lowest lower'

Input   : 'newest widest'
Tokens  : ['newest</w>', 'widest</w>']
Decoded : 'newest widest'

Input   : 'the cat sat on the mat'
Tokens  : ['t', '<unk>', 'e', '</w>', '<unk>', '<unk>', 't', '</w>', 's', '<unk>', 't', '</w>', 'o', 'n', '</w>', 't', '<unk>', 'e', '</w>', '<unk>', '<unk>', 't', '</w>']
Decoded : 'te t st on te t'



In [6]:
if __name__ == "__main__":
    corpus = "low lower newest widest lowest low low"

    tok = BPETokenizer()
    tok.train(corpus, num_merges=15)

    print("=== Vocabulary ===")
    for token, idx in sorted(tok.token_to_id.items(), key=lambda x: x[1]):
        print(f"  {idx:3d}  {token!r}")

    print("\n=== Merge rules ===")
    for i, rule in enumerate(tok.merge_rules, 1):
        print(f"  {i:2d}. {rule[0]!r} + {rule[1]!r} → {rule[0]+rule[1]!r}")

    print("\n=== Encode / Decode roundtrip ===")
    sentences = [
        """The roundtrip always works for in-vocabulary text because every merge is reversible. The decoder doesn't need to know the merge rules at all — it just concatenates strings. The information is all preserved in the token boundaries.
"""
    ]
    for s in sentences:
        tokens  = tok.tokenize(s)
        ids     = tok.encode(s)
        decoded = tok.decode(ids)
        match   = "✓" if decoded == s else "✗"
        print(f"  {match}  '{s}'")
        print(f"       tokens : {tokens}")
        print(f"       ids    : {ids}")
        print(f"       decoded: '{decoded}'")

=== Vocabulary ===
    0  'd'
    1  'e'
    2  'i'
    3  'l'
    4  'n'
    5  'o'
    6  'r'
    7  's'
    8  't'
    9  'w'
   10  'es'
   11  'lo'
   12  'ne'
   13  'wi'
   14  'est'
   15  'low'
   16  'new'
   17  'wid'
   18  '</w>'
   19  'lowe'
   20  'lower'
   21  'est</w>'
   22  'low</w>'
   23  'lower</w>'
   24  'newest</w>'
   25  'widest</w>'

=== Merge rules ===
   1. 'l' + 'o' → 'lo'
   2. 'lo' + 'w' → 'low'
   3. 'low' + '</w>' → 'low</w>'
   4. 'e' + 's' → 'es'
   5. 'es' + 't' → 'est'
   6. 'est' + '</w>' → 'est</w>'
   7. 'low' + 'e' → 'lowe'
   8. 'lowe' + 'r' → 'lower'
   9. 'lower' + '</w>' → 'lower</w>'
  10. 'n' + 'e' → 'ne'
  11. 'ne' + 'w' → 'new'
  12. 'new' + 'est</w>' → 'newest</w>'
  13. 'w' + 'i' → 'wi'
  14. 'wi' + 'd' → 'wid'
  15. 'wid' + 'est</w>' → 'widest</w>'

=== Encode / Decode roundtrip ===
  ✗  'The roundtrip always works for in-vocabulary text because every merge is reversible. The decoder doesn't need to know the merge rules at all — i